# Fase 5 — Benchmark FPS / Latensi (H3) — **versi Colab**

Mengukur latensi inferensi 8 varian arsitektur: **B2, BB-Deep, BB-Early, BB-Late-Alt,
BB-Late, NeckOnly, BackboneOnly, P1**.

## Syarat GPU (penting)

**Pilih runtime T4** (Runtime → Change runtime type → T4 GPU) agar sebanding dengan catatan
FPS lama di tracker (B2 ~51 FPS di T4). Notebook memberi peringatan kalau bukan T4.

Yang **benar-benar wajib**: kedelapan varian diukur dalam **SATU sesi, di GPU yang sama** —
karena yang dibandingkan adalah FPS *relatif* antar-varian. Sel benchmark mengukur semuanya
dalam satu loop, jadi syarat ini otomatis terpenuhi selama sesi tidak putus di tengah.
Kalau sesi putus, **ulangi dari awal** — jangan gabungkan hasil dari dua sesi berbeda.

**Bobot terlatih tidak diperlukan:** kecepatan ditentukan arsitektur, bukan nilai bobot. Model
dibangun dari YAML lalu di-`fuse()` (Conv+BN digabung, seperti kondisi deployment). Ini juga
membuat benchmark bisa diulang siapa pun tanpa akses checkpoint kita.

**Empat hal yang bikin angkanya bisa dipercaya (mudah terlewat):**
1. **Warm-up** — inferensi pertama memuat/compile kernel CUDA; tanpa ini angkanya bisa meleset 2-10x.
2. **`torch.cuda.synchronize()`** — panggilan CUDA *asynchronous*; tanpa sinkronisasi kita cuma
   mengukur waktu *melempar* pekerjaan ke GPU, bukan waktu GPU menyelesaikannya.
3. **Median dari 300 ulangan**, bukan rata-rata — tahan lonjakan sesekali (throttle, proses lain).
4. **batch=1 + fuse()** — konvensi pelaporan latensi di paper deteksi objek.


## 0. Mount Drive + Kredensial Kaggle

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, stat, shutil

DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs(DRIVE_ROOT, exist_ok=True)
KAGGLE_JSON_CACHE = f"{DRIVE_ROOT}/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)
assert os.path.exists(KAGGLE_JSON_CACHE), (
    f"kaggle.json tidak ada di {KAGGLE_JSON_CACHE} -- jalankan fase-setup-colab-mamba.ipynb dulu."
)
shutil.copy(KAGGLE_JSON_CACHE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
print("kaggle.json dimuat dari cache Drive.")

Mounted at /content/drive
kaggle.json dimuat dari cache Drive.


## 1. Setup Repo + Patch (identik notebook training)

In [2]:
import subprocess, sys, os, shutil, glob, torch

# TF32 dimatikan spt notebook training -- konsistensi numerik. (T4 memang tidak punya TF32,
# tapi baris ini dipertahankan supaya perilaku identik kalau notebook dijalankan di GPU lain.)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

DEV = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
print(f"GPU: {DEV}  (compute capability {CAP})")
print(f"torch {torch.__version__}")
if "T4" not in DEV:
    print(f"\n!!! PERINGATAN: GPU BUKAN T4 ({DEV}).")
    print("!!! Angka FPS TIDAK sebanding dgn hasil T4 lain. Untuk paper, jalankan di Kaggle T4.")

REPO = "/content/Mamba-YOLO"
# WAJIB chdir keluar SEBELUM rmtree: kalau notebook di-run ulang di kernel yg sama dan cwd
# sedang di dalam REPO, menghapusnya bikin semua subprocess berikutnya gagal dgn
# "fatal: Unable to read current working directory".
os.chdir("/content")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/HZAI-ZJNU/Mamba-YOLO.git", REPO], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
os.makedirs("/content/deform_src", exist_ok=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/deform-ss2d-py",
                "-p", "/content/deform_src", "--unzip"], check=True)
DEFORM_SRC = glob.glob("/content/deform_src/**/deform_ss2d.py", recursive=True)[0]
shutil.copy(DEFORM_SRC, f"{REPO}/ultralytics/nn/modules/deform_ss2d.py")
print(f"deform_ss2d.py disalin dari {DEFORM_SRC}")

GPU: Tesla T4  (compute capability (7, 5))
torch 2.11.0+cu128
deform_ss2d.py disalin dari /content/deform_src/deform_ss2d.py


In [3]:
import re, py_compile

def patch(path, subs):
    p = f"{REPO}/{path}"
    src = open(p).read()
    for old, new, tag in subs:
        if new in src:
            print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new); print(f"  OK {tag}")
    open(p, "w").write(src)

print("### Patch registrasi")
patch("ultralytics/nn/modules/mamba_yolo.py", [
    ("class XSSBlock(nn.Module):\n    def __init__(",
     "class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.1"),
    ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,",
     "nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,", "1.2"),
    ("class VSSBlock(nn.Module):\n    def __init__(",
     "class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.3"),
    ("            self.op = SS2D(", "            self.op = self.SSM_CLASS(", "1.4"),
])
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("from .mamba_yolo import SS2D", "from .mamba_yolo import SS2D, VSSBlock, XSSBlock", "2.1"),
])
patch("ultralytics/nn/tasks.py", [
    ("    XSSBlock\n)\n",
     "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import DeformVSSBlock, DeformXSSBlock\n", "3.1"),
    ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
     "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock\n        }:", "3.2"),
    ("RepC3, XSSBlock}:", "RepC3, XSSBlock, DeformXSSBlock}:", "3.3"),
])

# --- Generate SEMUA YAML varian (identik logika notebook Ablasi B) ---
base_yaml = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()

y = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", re.sub(r"\bVSSBlock\b", "DeformVSSBlock", base_yaml))
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml", "w").write(y)

idx = base_yaml.find("# Mamba-YOLO PAFPN")
assert idx != -1
bb_part, head_part = base_yaml[:idx], base_yaml[idx:]
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-backbone-only.yaml", "w").write(
    re.sub(r"\bVSSBlock\b", "DeformVSSBlock", bb_part) + head_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-neck-only.yaml", "w").write(
    bb_part + re.sub(r"\bXSSBlock\b", "DeformXSSBlock", head_part))

def _swap_positions(text, positions):
    lines = text.split("\n"); hit = {p: 0 for p in positions}
    for i, line in enumerate(lines):
        for p in positions:
            if line.rstrip().endswith(f"# {p}") and "VSSBlock" in line and "Deform" not in line:
                lines[i] = line.replace("VSSBlock", "DeformVSSBlock"); hit[p] += 1
    for p in positions:
        assert hit[p] == 1, f"posisi {p}: {hit[p]}x"
    return "\n".join(lines)

for vname, pos in {"bb-deep": [7], "bb-late": [5, 7], "bb-early": [1, 3]}.items():
    open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-{vname}.yaml", "w").write(
        _swap_positions(base_yaml, pos))

# BB-Late-Alt: posisi 5 dipecah D-S-D + reindex head (+2)
_l = base_yaml.split("\n")
_t5 = next(i for i, l in enumerate(_l) if l.rstrip().endswith("# 5") and "VSSBlock" in l and "Deform" not in l)
_l = _l[:_t5] + ["  - [-1, 1, DeformVSSBlock, [512]]      # 5a (D)",
                 "  - [-1, 1, VSSBlock, [512]]             # 5b (S)",
                 "  - [-1, 1, DeformVSSBlock, [512]]      # 5c (D)"] + _l[_t5+1:]
for i, l in enumerate(_l):
    if l.rstrip().endswith("# 7") and "VSSBlock" in l and "Deform" not in l:
        _l[i] = l.replace("VSSBlock", "DeformVSSBlock")
_txt = "\n".join(_l)
for _o, _n in [("  - [[-1, 5], 1, Concat, [1]]  # cat backbone P4", "  - [[-1, 7], 1, Concat, [1]]  # cat backbone P4"),
               ("  - [[-1, 11], 1, Concat, [1]]  # cat head P4", "  - [[-1, 13], 1, Concat, [1]]  # cat head P4"),
               ("  - [[-1, 8], 1, Concat, [1]]  # cat head P5", "  - [[-1, 10], 1, Concat, [1]]  # cat head P5"),
               ("  - [[14, 17, 20], 1, Detect, [nc]]  # Detect(P3, P4, P5)",
                "  - [[16, 19, 22], 1, Detect, [nc]]  # Detect(P3, P4, P5)")]:
    assert _txt.count(_o) == 1, f"reindex gagal: {_o!r}"
    _txt = _txt.replace(_o, _n)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-bb-late-alt.yaml", "w").write(_txt)
print("  OK semua YAML varian digenerate")

for f in ["ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py",
          "ultralytics/nn/tasks.py"]:
    py_compile.compile(f"{REPO}/{f}", doraise=True)
print("  OK sintaks valid")

### Patch registrasi
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
  ~ 2.1: skip
  OK 3.1
  OK 3.2
  OK 3.3
  OK semua YAML varian digenerate
  OK sintaks valid


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)
os.environ["MAX_JOBS"] = str(os.cpu_count() or 2)

# Cache wheel selective_scan di Drive, ditandai arsitektur GPU + versi torch/python.
# Kompilasi CUDA ini satu-satunya langkah yg lambat (beberapa menit); sesi berikutnya
# dgn GPU sama langsung install dari cache. Wheel TIDAK portabel antar-arsitektur GPU,
# makanya tag-nya menyertakan compute capability.
cap = torch.cuda.get_device_capability(0)
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"
os.makedirs(SS_WHEEL_DIR, exist_ok=True)

cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
if cached_ss:
    print(f"Wheel cache ditemukan ({SS_TAG}) -- install langsung, skip compile")
    subprocess.run([sys.executable, "-m", "pip", "install"] + cached_ss, check=True)
else:
    print(f"Belum ada cache utk {SS_TAG} -- compile sekali ini, lalu disimpan ke Drive")
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-v", "-w", SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    subprocess.run([sys.executable, "-m", "pip", "install"] + glob.glob(f"{SS_WHEEL_DIR}/*.whl"), check=True)
    print(f"Tersimpan ke {SS_WHEEL_DIR}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn", "thop", "timm", "einops"], check=True)

for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]:
    del sys.modules[k]
import selective_scan_cuda_core
print("selective_scan_cuda_core OK -- setup selesai.")

Belum ada cache utk sm75_torch2.11.0_py313 -- compile sekali ini, lalu disimpan ke Drive
Tersimpan ke /content/drive/MyDrive/demamba-yolowd/wheels/selective_scan_sm75_torch2.11.0_py313
selective_scan_cuda_core OK -- setup selesai.


## 2. Benchmark

**Parameter yang dilaporkan di paper** (semuanya dicetak di bawah): GPU, presisi, `imgsz`,
`batch`, jumlah warm-up, jumlah pengulangan.

In [ ]:
import numpy as np, pandas as pd, time, json
os.chdir(REPO); sys.path.insert(0, REPO)
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules.deform_ss2d import DeformSS2D

IMGSZ, BATCH, NC = 640, 1, 8      # nc=8 (DSF-YOLO); batch=1 = konvensi latensi paper
WARMUP, REPEAT   = 50, 300        # warm-up WAJIB; 300 ulangan utk sebaran yg stabil
USE_AMP          = False          # FP32. Set True utk ukur skenario AMP (laporkan terpisah!)

VARIANTS = {
    "B2":           "Mamba-YOLO-T.yaml",
    "BB-Deep":      "DeMamba-YOLO-T-bb-deep.yaml",
    "BB-Early":     "DeMamba-YOLO-T-bb-early.yaml",
    "BB-Late-Alt":  "DeMamba-YOLO-T-bb-late-alt.yaml",
    "BB-Late":      "DeMamba-YOLO-T-bb-late.yaml",
    "NeckOnly":     "DeMamba-YOLO-T-neck-only.yaml",
    "BackboneOnly": "DeMamba-YOLO-T-backbone-only.yaml",
    "P1":           "DeMamba-YOLO-T.yaml",
}

def bench(cfg_name):
    m = DetectionModel(f"ultralytics/cfg/models/mamba-yolo/{cfg_name}", ch=3, nc=NC, verbose=False)
    n_par = sum(p.numel() for p in m.parameters())
    n_def = sum(1 for x in m.modules() if isinstance(x, DeformSS2D))
    m = m.fuse() if hasattr(m, "fuse") else m      # Conv+BN digabung, spt deployment nyata
    m = m.cuda().eval()
    x = torch.randn(BATCH, 3, IMGSZ, IMGSZ, device="cuda")

    with torch.no_grad(), torch.autocast("cuda", enabled=USE_AMP):
        for _ in range(WARMUP):                     # (1) warm-up: kompilasi/cache kernel CUDA
            m(x)
        torch.cuda.synchronize()                    # (2) tunggu GPU benar-benar selesai

        lat = []
        for _ in range(REPEAT):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            m(x)
            torch.cuda.synchronize(); lat.append((time.perf_counter() - t0) * 1000.0)

    del m, x; torch.cuda.empty_cache()
    a = np.array(lat)
    return {"params_M": n_par/1e6, "n_deform": n_def,
            "lat_median_ms": float(np.median(a)), "lat_mean_ms": float(a.mean()),
            "lat_std_ms": float(a.std(ddof=1)), "lat_p90_ms": float(np.percentile(a, 90)),
            "fps_median": 1000.0/float(np.median(a))}

rows = []
for name, cfg in VARIANTS.items():
    r = bench(cfg); r["variant"] = name; rows.append(r)
    print(f"{name:14s} {r['params_M']:.3f}M  {r['n_deform']:2d} def  "
          f"{r['lat_median_ms']:6.2f} ms  {r['fps_median']:6.1f} FPS")

df = pd.DataFrame(rows)[["variant","n_deform","params_M","lat_median_ms","lat_mean_ms",
                          "lat_std_ms","lat_p90_ms","fps_median"]]
b2_lat = df.loc[df.variant == "B2", "lat_median_ms"].iloc[0]
df["slowdown_vs_B2"] = df.lat_median_ms / b2_lat
df["fps_pct_of_B2"]  = 100.0 * b2_lat / df.lat_median_ms

META = {"gpu": DEV, "compute_capability": str(CAP), "torch": torch.__version__,
        "imgsz": IMGSZ, "batch": BATCH, "precision": "AMP" if USE_AMP else "FP32",
        "warmup": WARMUP, "repeat": REPEAT}
print("\n=== KONDISI PENGUKURAN (WAJIB disebut di paper) ===")
for k, v in META.items(): print(f"  {k}: {v}")
print()
print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

OUT_DIR = f"{DRIVE_ROOT}/analysis"
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(f"{OUT_DIR}/fps_benchmark.csv", index=False)
json.dump(META, open(f"{OUT_DIR}/fps_benchmark_meta.json", "w"), indent=2)
# salinan lokal juga, biar gampang diunduh langsung dari panel file Colab
df.to_csv("/content/fps_benchmark.csv", index=False)
print(f"\nTersimpan ke Drive: {OUT_DIR}/fps_benchmark.csv (+ _meta.json)")
print("Salinan lokal: /content/fps_benchmark.csv")

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd


Mamba-YOLO-T summary: 388 layers, 5985568 parameters, 5985552 gradients, 13.6 GFLOPs


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:975: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  param_grad = param.grad


B2             5.987M   0 def   23.10 ms    43.3 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(


DeMamba-YOLO-T-bb-deep summary: 402 layers, 5989374 parameters, 5989358 gradients, 13.7 GFLOPs
BB-Deep        5.991M   1 def   26.02 ms    38.4 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
DeMamba-YOLO-T-bb-early summary: 416 layers, 5973715 parameters, 5973699 gradients, 13.7 GFLOPs
BB-Early       5.975M   2 def   28.28 ms    35.4 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
DeMamba-YOLO-T-bb-late-alt summary: 429 layers, 5971600 parameters, 5971584 gradients, 13.7 GFLOPs
BB-Late-Alt    5.973M   3 def   30.75 ms    32.5 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
DeMamba-YOLO-T-bb-late summary: 444 layers, 5962713 parameters, 5962697 gradients, 13.8 GFLOPs
BB-Late        5.964M   4 def   32.11 ms    31.1 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


DeMamba-YOLO-T-neck-only summary: 444 layers, 5964233 parameters, 5964217 gradients, 13.8 GFLOPs
NeckOnly       5.966M   4 def   29.02 ms    34.5 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
DeMamba-YOLO-T-backbone-only summary: 472 layers, 5950860 parameters, 5950844 gradients, 13.9 GFLOPs
BackboneOnly   5.952M   6 def   33.12 ms    30.2 FPS
Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.
DeMamba-YOLO-T summary: 528 layers, 5929525 parameters, 5929509 gradients, 14.1 GFLOPs
P1             5.931M  10 def   38.75 ms    25.8 FPS

=== KONDISI PENGUKURAN (WAJIB disebut di paper) ===
  gpu: Tesla T4
  compute_capability: (7, 5)
  torch: 2.11.0+cu128
  imgsz: 640
  batch: 1
  precision: FP32
  warmup: 50
  repeat: 300

     variant  n_deform  params_M  lat_median_ms  lat_mean_ms  lat_std_ms  lat_p90_ms  fps_median  slowdown_vs_B2  fps_pct_of_B2
          B2         0     5.987         23.097       

## 3. Ringkasan untuk H3

Baris **`slowdown_vs_B2`** adalah angka inti H3: berapa kali lebih lambat tiap varian dibanding
baseline. **`fps_pct_of_B2`** = FPS varian sebagai persentase FPS B2 (100% = sama cepat).

Hasil tersimpan di Drive (`demamba-yolowd/analysis/fps_benchmark.csv`) dan `/content/fps_benchmark.csv`.
Kirim CSV itu ke sesi analisis untuk digabung ke tracker.

In [ ]:
print("=== RINGKASAN H3: biaya efisiensi deformable scanning ===\n")
s = df.sort_values("n_deform")
for _, r in s.iterrows():
    bar = "#" * int(round((r.slowdown_vs_B2 - 1) * 100))
    print(f"{r.variant:14s} {int(r.n_deform):2d} blok deform  "
          f"{r.lat_median_ms:6.2f} ms  {r.fps_median:6.1f} FPS  "
          f"{r.slowdown_vs_B2:5.2f}x  {bar}")

p1 = df[df.variant == "P1"].iloc[0]
print(f"\nP1 (deformable penuh) vs B2: {p1.slowdown_vs_B2:.2f}x lebih lambat "
      f"({p1.fps_pct_of_B2:.1f}% FPS B2)")
print("\nCatatan: bandingkan kolom slowdown terhadap selisih akurasi (semua TIDAK signifikan,")
print("lihat tracker Section 6.3c/6.3d) -- inti argumen H3: biaya nyata, manfaat tidak terbukti.")

## 4. Matikan Runtime (Jalankan Terakhir)

In [ ]:
# Matikan runtime Colab otomatis setelah semua sel di atas selesai --
# supaya TIDAK terus memakan kuota Colab Pro kalau kamu tidak sempat matikan manual.
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)  # jeda singkat, jaga-jaga ada file yang masih proses ditulis ke Drive

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")